# 106 --- Live Option Chain (In-Notebook)

A fully self-contained, live-updating option chain that renders **inside the notebook**.
No Streamlit, no external browser app. Uses `ipywidgets` for interactive controls
and `IPython.display` for live-updating styled tables.

**Features:**
- Dropdown expiration selector
- Full option chain with Greeks (IV, Delta, Gamma, Theta) computed via the Rust Black-Scholes calculator
- ITM/ATM color coding
- One-click and auto-refresh modes
- IV smile plot and term structure visualization

**Install dependencies:**
```
pip install thetadatadx[pandas] ipywidgets
```

In [ ]:
from thetadatadx import Credentials, Config, ThetaDataDx, all_greeks, implied_volatility
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

creds = Credentials("your_email@example.com", "your_password")
tdx = ThetaDataDx(creds, Config.production())
print("Connected!")

## Choose Ticker and Expiration

In [ ]:
ticker = "SPY"

# Fetch spot price
spot_snap = tdx.stock_snapshot_ohlc([ticker])
spot = spot_snap[0]["close"] if spot_snap else 0.0
print(f"{ticker} spot: ${spot:.2f}")

# Fetch expirations and build readable labels
expirations = tdx.option_list_expirations(ticker)
today = datetime.now()

exp_labels = []
for exp_str in expirations[:20]:
    exp_dt = datetime.strptime(exp_str, "%Y%m%d")
    dte = (exp_dt - today).days
    exp_labels.append(f"{exp_str}  ({exp_dt.strftime('%b %d')} --- {dte}d)")

print(f"{ticker}: {len(expirations)} expirations available")

# Dropdown for expiration
exp_dropdown = widgets.Dropdown(
    options=list(zip(exp_labels, expirations[:20])),
    description="Expiration:",
    style={"description_width": "initial"},
)
display(exp_dropdown)

## Build the Option Chain

In [ ]:
RISK_FREE = 0.05
DIV_YIELD = 0.013


def build_option_chain(ticker, expiration, spot_price, num_strikes=20):
    """Fetch quotes for calls and puts, compute Greeks, return DataFrame."""

    exp_dt = datetime.strptime(expiration, "%Y%m%d")
    dte = (exp_dt - datetime.now()).days
    tte = max(dte / 365.0, 1 / 365.0)

    # Get all strikes
    all_strikes = tdx.option_list_strikes(ticker, expiration)
    if not all_strikes:
        return pd.DataFrame()

    # ThetaData strikes are stored as integer strings (strike * 1000)
    strike_pairs = [(s, float(s) / 1000) for s in all_strikes]

    # Filter to N strikes around ATM
    atm_idx = min(range(len(strike_pairs)), key=lambda i: abs(strike_pairs[i][1] - spot_price))
    half = num_strikes // 2
    start = max(0, atm_idx - half)
    end = min(len(strike_pairs), atm_idx + half + 1)
    selected = strike_pairs[start:end]

    rows = []
    for strike_str, strike_val in selected:
        row = {"strike": strike_val}

        for right, prefix in [("C", "call_"), ("P", "put_")]:
            is_call = (right == "C")

            # Quote (bid/ask)
            try:
                quote = tdx.option_snapshot_quote(ticker, expiration, strike_str, right)
                if quote:
                    bid = quote[0]["bid"]
                    ask = quote[0]["ask"]
                    mid = (bid + ask) / 2
                else:
                    bid = ask = mid = np.nan
            except Exception:
                bid = ask = mid = np.nan

            row[f"{prefix}bid"] = bid
            row[f"{prefix}ask"] = ask

            # Last trade
            try:
                trade = tdx.option_snapshot_trade(ticker, expiration, strike_str, right)
                row[f"{prefix}last"] = trade[0]["price"] if trade else np.nan
                row[f"{prefix}volume"] = trade[0].get("size", 0) if trade else 0
            except Exception:
                row[f"{prefix}last"] = np.nan
                row[f"{prefix}volume"] = 0

            # Open interest
            try:
                oi_data = tdx.option_snapshot_open_interest(ticker, expiration, strike_str, right)
                if oi_data:
                    oi_val = oi_data[0].get("open_interest", 0)
                    if isinstance(oi_val, dict):
                        oi_val = oi_val.get("value", 0)
                    row[f"{prefix}oi"] = int(oi_val) if oi_val else 0
                else:
                    row[f"{prefix}oi"] = 0
            except Exception:
                row[f"{prefix}oi"] = 0

            # Greeks via Rust Black-Scholes
            if pd.notna(mid) and mid > 0.01:
                try:
                    g = all_greeks(
                        spot=spot_price,
                        strike=strike_val,
                        rate=RISK_FREE,
                        div_yield=DIV_YIELD,
                        tte=tte,
                        option_price=mid,
                        is_call=is_call,
                    )
                    if g["iv_error"] < 0.05:
                        row[f"{prefix}iv"] = g["iv"]
                        row[f"{prefix}delta"] = g["delta"]
                        row[f"{prefix}gamma"] = g["gamma"]
                        row[f"{prefix}theta"] = g["theta"]
                    else:
                        for k in ("iv", "delta", "gamma", "theta"):
                            row[f"{prefix}{k}"] = np.nan
                except Exception:
                    for k in ("iv", "delta", "gamma", "theta"):
                        row[f"{prefix}{k}"] = np.nan
            else:
                for k in ("iv", "delta", "gamma", "theta"):
                    row[f"{prefix}{k}"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def style_chain(df, spot_price):
    """Apply option-chain styling: ITM highlighting, ATM gold, formatted numbers."""

    # Column order: Calls | Strike | Puts
    call_cols = [
        "call_iv", "call_delta", "call_gamma", "call_theta",
        "call_bid", "call_ask", "call_last", "call_volume", "call_oi",
    ]
    put_cols = [
        "put_bid", "put_ask", "put_last", "put_volume", "put_oi",
        "put_iv", "put_delta", "put_gamma", "put_theta",
    ]
    display_cols = [c for c in call_cols if c in df.columns]
    display_cols.append("strike")
    display_cols.extend([c for c in put_cols if c in df.columns])

    styled_df = df[display_cols].copy()

    # Rename for cleaner headers
    rename = {}
    for c in styled_df.columns:
        parts = c.split("_", 1)
        if len(parts) == 2 and parts[0] in ("call", "put"):
            rename[c] = f"{parts[0][0].upper()}_{parts[1].upper()}"
        else:
            rename[c] = c.upper()
    styled_df.columns = [rename.get(c, c) for c in styled_df.columns]

    # ATM strike
    atm_strike = df.loc[(df["strike"] - spot_price).abs().idxmin(), "strike"]

    def highlight_itm(row):
        styles = [""] * len(row)
        if "STRIKE" not in row.index:
            return styles
        strike = row["STRIKE"]
        strike_pos = list(row.index).index("STRIKE")

        # ITM calls: strike < spot
        if strike < spot_price:
            for i in range(strike_pos):
                styles[i] = "background-color: rgba(76, 175, 80, 0.15)"
        # ITM puts: strike > spot
        if strike > spot_price:
            for i in range(strike_pos + 1, len(row)):
                styles[i] = "background-color: rgba(76, 175, 80, 0.15)"
        # ATM
        if strike == atm_strike:
            styles[strike_pos] = "background-color: rgba(255, 215, 0, 0.35); font-weight: bold"
        return styles

    # Format dict
    fmt = {}
    for c in styled_df.columns:
        if "IV" in c:
            fmt[c] = lambda x: f"{x*100:.1f}%" if pd.notna(x) else ""
        elif "DELTA" in c or "GAMMA" in c or "THETA" in c:
            fmt[c] = lambda x: f"{x:.4f}" if pd.notna(x) else ""
        elif "VOLUME" in c or "OI" in c:
            fmt[c] = lambda x: f"{int(x):,}" if pd.notna(x) and x != 0 else ""
        elif c == "STRIKE":
            fmt[c] = lambda x: f"${x:.1f}" if pd.notna(x) else ""
        else:
            fmt[c] = lambda x: f"{x:.2f}" if pd.notna(x) else ""

    return styled_df.style.apply(highlight_itm, axis=1).format(fmt)


print("build_option_chain() and style_chain() defined.")

In [ ]:
chain = build_option_chain(ticker, exp_dropdown.value, spot)
exp_dt = datetime.strptime(exp_dropdown.value, "%Y%m%d")
dte = (exp_dt - datetime.now()).days

display(HTML(
    f"<h3>{ticker} Option Chain &mdash; {exp_dropdown.value} "
    f"({exp_dt.strftime('%b %d, %Y')}, {dte} DTE) &mdash; "
    f"Spot ${spot:.2f}</h3>"
))
display(style_chain(chain, spot))

## Live Auto-Refresh

Use the **Refresh** button for a one-shot update, or toggle **Auto-refresh**
and run the next cell to poll every 5 seconds. Interrupt the kernel to stop.

In [ ]:
output = widgets.Output()
refresh_btn = widgets.Button(description="Refresh", button_style="info", icon="refresh")
auto_toggle = widgets.ToggleButton(value=False, description="Auto-refresh (5s)", icon="repeat")
strikes_slider = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Strikes:")
controls = widgets.HBox([refresh_btn, auto_toggle, strikes_slider])


def refresh(_=None):
    with output:
        clear_output(wait=True)

        # Re-fetch spot
        snap = tdx.stock_snapshot_ohlc([ticker])
        current_spot = snap[0]["close"] if snap else spot

        chain = build_option_chain(
            ticker, exp_dropdown.value, current_spot,
            num_strikes=strikes_slider.value,
        )

        if chain.empty:
            print("No data returned.")
            return

        exp_dt = datetime.strptime(exp_dropdown.value, "%Y%m%d")
        dte = (exp_dt - datetime.now()).days
        ts = time.strftime("%H:%M:%S")

        display(HTML(
            f"<div style='display:flex; justify-content:space-between; align-items:baseline;'>"
            f"<span style='color:#4FC3F7; font-weight:bold; font-size:14px;'>CALLS</span>"
            f"<span style='font-size:13px;'>{ticker} &mdash; {exp_dropdown.value} "
            f"({dte} DTE) &mdash; Spot ${current_spot:.2f} &mdash; {ts}</span>"
            f"<span style='color:#FF8A65; font-weight:bold; font-size:14px;'>PUTS</span>"
            f"</div>"
        ))
        display(style_chain(chain, current_spot))


refresh_btn.on_click(refresh)
display(controls, output)
refresh()

In [ ]:
# Run this cell to start auto-refresh. Interrupt the kernel (stop button) to stop.
while auto_toggle.value:
    refresh()
    time.sleep(5)

## IV Smile Visualization

Plot the implied volatility smile for calls and puts across strikes
for the currently selected expiration.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

chain = build_option_chain(ticker, exp_dropdown.value, spot, num_strikes=30)

fig, ax = plt.subplots(figsize=(12, 5))

valid_call = chain.dropna(subset=["call_iv"])
valid_put = chain.dropna(subset=["put_iv"])

if len(valid_call) > 0:
    ax.plot(
        valid_call["strike"], valid_call["call_iv"] * 100,
        "o-", color="#1f77b4", markersize=4, linewidth=1.2, label="Call IV",
    )
if len(valid_put) > 0:
    ax.plot(
        valid_put["strike"], valid_put["put_iv"] * 100,
        "s-", color="#d62728", markersize=4, linewidth=1.2, label="Put IV",
    )

ax.axvline(x=spot, color="gray", linestyle="--", linewidth=1, alpha=0.7, label=f"Spot ${spot:.0f}")
ax.set_xlabel("Strike ($)")
ax.set_ylabel("Implied Volatility (%)")

exp_dt = datetime.strptime(exp_dropdown.value, "%Y%m%d")
dte = (exp_dt - datetime.now()).days
ax.set_title(f"{ticker} IV Smile --- {exp_dropdown.value} ({dte} DTE)")
ax.legend()
plt.tight_layout()
plt.show()

## Multi-Expiration Term Structure

Fetch the ATM implied volatility for each of the nearest expirations
to visualize how IV varies across the term structure.

In [ ]:
atm_ivs = []
for exp_str in expirations[:10]:
    try:
        ch = build_option_chain(ticker, exp_str, spot, num_strikes=3)
        if ch.empty or "call_iv" not in ch.columns:
            continue

        # Pick the row closest to ATM
        atm_row = ch.loc[(ch["strike"] - spot).abs().idxmin()]
        call_iv = atm_row.get("call_iv", np.nan)
        put_iv = atm_row.get("put_iv", np.nan)

        if pd.notna(call_iv) and call_iv > 0:
            exp_dt = datetime.strptime(exp_str, "%Y%m%d")
            dte = (exp_dt - datetime.now()).days
            atm_ivs.append({
                "Expiration": exp_str,
                "DTE": dte,
                "Call ATM IV": call_iv,
                "Put ATM IV": put_iv if pd.notna(put_iv) else np.nan,
            })
            print(f"  {exp_str} ({dte:>3}d): Call IV {call_iv*100:.1f}%")
    except Exception as exc:
        print(f"  {exp_str}: skipped ({exc})")
        continue

if atm_ivs:
    term = pd.DataFrame(atm_ivs)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(term["DTE"], term["Call ATM IV"] * 100, "o-", color="#1f77b4", linewidth=2, label="Call ATM IV")
    if term["Put ATM IV"].notna().any():
        ax.plot(term["DTE"], term["Put ATM IV"] * 100, "s-", color="#d62728", linewidth=2, label="Put ATM IV")
    ax.set_xlabel("Days to Expiration")
    ax.set_ylabel("ATM Implied Volatility (%)")
    ax.set_title(f"{ticker} IV Term Structure")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No ATM IV data collected.")

---

**Full dashboard:** For a richer real-time GUI with tabbed expirations, open-interest charts,
and configurable display, run the Streamlit app:

```
streamlit run ../tools/live-chain/app.py
```

**Previous:** [105 --- Real-Time Streaming with FPSS](105_realtime_streaming.ipynb)  
**Back to start:** [101 --- Getting Started](101_getting_started.ipynb)